# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faja27/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal check #1 (flag-linked — volume behind `is_quick_win`)

claim: for pages sitting in *striking distance* (position 11-20, one step from page 1), higher search volume (`impressions_90d`) actually turns into meaningfully more clicks — i.e. the "quick win" is only worth chasing when there's real demand behind it, not just a good position.

test: bucket `clicks_90d` by `impression_tier`, restricted to `position_tier == "striking"`, with n printed (code cell below).

verdict: **CONFIRMED** — median clicks climb 0 → 1 → 15 → 134 from `low` to `excellent` impression tier, on n=91–3,325 per bucket (well above the 50-row floor). Volume is real, not noise sitting next to a decent rank.

### Signal check #2 (position vs. CTR — supports the "striking distance" half of the rule)

claim: average CTR should fall off as position gets worse, which is *why* striking distance (11-20) is worth prioritising over deeper pages (`page_3_5`, `deep`) for a push toward page 1.

test: weighted CTR (`sum(clicks)/sum(impressions)`, not the mean of per-row rates — averaging per-row rates would misrepresent the true rate) by `position_tier`, with n printed (code cell below).

verdict: MIXED — the overall shape is right: CTR falls off a cliff past striking distance (0.35% at `striking` vs 0.15% at `page_3_5` vs 0.04% at `deep`), real support for excluding anything past striking distance. But it's not a clean CONFIRMED: `page_1` (0.350%) and `striking` (0.347%) are almost identical, so "closer rank = more clicks" doesn't hold at that particular boundary — moving a page from position 15 to 8 by itself probably won't move CTR much. The real reward is reaching `top_3` (0.49%). I keep this signal in the rule anyway, because the cliff at `page_3_5` `deep` is the part I actually rely on (to *exclude* those pages, not to rank *within* page_1/striking).

### My rule, in plain words

A page is a*quick-win candidate if it's already close to page 1 (`position_tier` is `page_1` or `striking` where signal #2 says the falloff hasn't hit yet) **and** it has real search demand behind it (`impression_tier`
is `moderate` or better — where signal #1 says the demand actually converts to clicks). Among candidates, the score is just the impression volume itself — bigger audience behind the same position = fix it first.

- **score:** `impressions_90d` if both conditions hold, else `0`
- **reason code:** `quick_win_candidate` (else `not_quick_win`)
- **action label:** `boost_ranking_priority` (else `no_action`)

One rule, one reason code, one action label — on purpose, so it's readable and honestly beatable next week.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faja27/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found - am i at the repo root?"

import pandas as pd
import numpy as np

pd.set_option("display.width", 140)
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape, "rows x cols loaded")

# --- signal check #1 (flag-linked, volume behind is_quick_win) ---
striking = df[df["position_tier"] == "striking"].copy()
sig1 = (striking.groupby("impression_tier")
        .agg(n=("content_id", "size"),
             median_clicks=("clicks_90d", "median"),
             mean_clicks=("clicks_90d", "mean"))
        .reindex(["low", "moderate", "good", "excellent"]))
print(f"\n--- signal 1: striking-distance rows n={len(striking)} ---")
print(sig1)
assert (sig1["n"] >= 50).all(), "a bucket is under the 50-row floor - verdict would be noise"
verdict_1 = "CONFIRMED"
print(f"VERDICT: {verdict_1}")

# --- signal check #2 (position vs weighted CTR) ---
order = ["top_3", "page_1", "striking", "page_3_5", "deep"]
sub = df[df["position_tier"].isin(order)]
rows = []
for tier in order:
    d = sub[sub["position_tier"] == tier]
    rows.append({
        "position_tier": tier,
        "n": len(d),
        "sum_impressions": d["impressions_90d"].sum(),
        "sum_clicks": d["clicks_90d"].sum(),
        "weighted_ctr_pct": round(100 * d["clicks_90d"].sum() / d["impressions_90d"].sum(), 3),
    })
sig2 = pd.DataFrame(rows).set_index("position_tier")
print(f"\n--- signal 2: position_tier vs weighted CTR ---")
print(sig2)
assert (sig2["n"] >= 50).all()
verdict_2 = "MIXED"
print(f"VERDICT: {verdict_2}")

# --- encode the rule: score, reason code, action label ---
in_range = df["position_tier"].isin(["page_1", "striking"])
has_demand = df["impression_tier"].isin(["moderate", "good", "excellent"])
is_candidate = in_range & has_demand

df["quick_win_score"] = np.where(is_candidate, df["impressions_90d"], 0)
df["reason_code"] = np.where(is_candidate, "quick_win_candidate", "not_quick_win")
df["action"] = np.where(is_candidate, "boost_ranking_priority", "no_action")

print(f"\ncandidates: {is_candidate.sum():,} of {len(df):,} rows ({is_candidate.mean():.1%})")
print(df.loc[is_candidate, ["position_tier", "impression_tier"]].value_counts())


Working dir: /content/flyrank-ml-internship
(30000, 44) rows x cols loaded

--- signal 1: striking-distance rows n=7304 ---
                    n  median_clicks  mean_clicks
impression_tier                                  
low              2226            0.0     0.235849
moderate         3325            1.0     2.721805
good             1662           15.0    30.308664
excellent          91          134.0   217.648352
VERDICT: CONFIRMED

--- signal 2: position_tier vs weighted CTR ---
                   n  sum_impressions  sum_clicks  weighted_ctr_pct
position_tier                                                      
top_3           2321          7032960       34355             0.488
page_1         11814         89575437      313804             0.350
striking        7304         22992054       79754             0.347
page_3_5        7242         35182261       54499             0.155
deep            1319          1228277         508             0.041
VERDICT: MIXED

candidates: 12,7

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Rank by `quick_win_score` descending, keep the columns a reviewer actually needs, write it from the notebook.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

output_cols = [
    "content_id", "client_id", "position_tier", "avg_position",
    "impression_tier", "impressions_90d", "clicks_90d", "ctr",
    "quick_win_score", "reason_code", "action",
]

ranked_queue = (df[output_cols]
                .sort_values("quick_win_score", ascending=False)
                .reset_index(drop=True))
ranked_queue.insert(0, "rank", ranked_queue.index + 1)

os.makedirs("work/outputs", exist_ok=True)
out_path = "work/outputs/baseline_action_score.csv"
ranked_queue.to_csv(out_path, index=False)

print(f"wrote {len(ranked_queue):,} rows to {out_path}")
ranked_queue.head(10)

wrote 30,000 rows to work/outputs/baseline_action_score.csv


,rank,content_id,client_id,position_tier,avg_position,impression_tier,impressions_90d,clicks_90d,ctr,quick_win_score,reason_code,action
0,1,content_5fe46e04994d,client_4e07408562,page_1,4.2,excellent,517715,741,0.14,517715,quick_win_candidate,boost_ranking_priority
1,2,content_aaef01a50def,client_19581e27de,page_1,5.4,excellent,517109,1270,0.25,517109,quick_win_candidate,boost_ranking_priority
2,3,content_1a9e894be2e2,client_19581e27de,page_1,4.0,excellent,416180,944,0.23,416180,quick_win_candidate,boost_ranking_priority
3,4,content_2c2606c5d176,client_19581e27de,page_1,4.2,excellent,347399,1854,0.53,347399,quick_win_candidate,boost_ranking_priority
4,5,content_db5989a78dd3,client_4e07408562,page_1,5.4,excellent,345111,733,0.21,345111,quick_win_candidate,boost_ranking_priority
5,6,content_cb112fce36be,client_19581e27de,page_1,5.6,excellent,309910,492,0.16,309910,quick_win_candidate,boost_ranking_priority
6,7,content_36ff89c8214e,client_19581e27de,page_1,7.3,excellent,295097,154,0.05,295097,quick_win_candidate,boost_ranking_priority
7,8,content_8e7ba84a972b,client_7f2253d7e2,page_1,4.8,excellent,288426,2653,0.92,288426,quick_win_candidate,boost_ranking_priority
8,9,content_89e84d699e9e,client_349c41201b,page_1,4.8,excellent,275226,2460,0.89,275226,quick_win_candidate,boost_ranking_priority
9,10,content_aa4baf490b43,client_349c41201b,page_1,5.9,excellent,256290,1276,0.50,256290,quick_win_candidate,boost_ranking_priority


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

(the assignment card asks for the top **10**, not 20 — a top-20 pass is optional/bonus, not required here)

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

benchmark_ctr = sig2.loc["striking", "weighted_ctr_pct"]  # ~0.35%, the pool this page is competing in
top10 = ranked_queue.head(10).copy()

def review_line(row):
    why = (f"{row.impressions_90d:,.0f} impressions/90d at avg position {row.avg_position:.1f} "
           f"({row.position_tier}) with {row.ctr:.2f}% CTR")
    if row.ctr < benchmark_ctr * 0.7:
        confidence = "low confidence - CTR gap may be intent/relevance, not just rank"
        risk = (f"CTR is well below the {benchmark_ctr:.2f}% benchmark for its tier - if that's "
                f"because the query/intent match is wrong (not just rank), pushing rank won't fix it")
    elif row.ctr > benchmark_ctr * 1.3:
        confidence = "medium confidence - page may already be near its ceiling"
        risk = (f"CTR ({row.ctr:.2f}%) is already ABOVE the {benchmark_ctr:.2f}% benchmark for its tier - "
                f"this page may already be near its ceiling, so the real upside is smaller than the raw "
                f"impression volume suggests")
    else:
        confidence = "high confidence - typical page for its rank"
        risk = (f"CTR is close to the {benchmark_ctr:.2f}% tier benchmark - a normal page for its rank, "
                f"so the score here is really just volume, not an efficiency problem")
    return why, confidence, risk

print(f"benchmark (striking-distance weighted CTR): {benchmark_ctr:.3f}%\n")
for i, row in top10.iterrows():
    why, confidence, risk = review_line(row)
    print(f"{row['rank']:2d}. {row.content_id} | action: {row.action} | reason: {row.reason_code}")
    print(f"    why:        {why}")
    print(f"    confidence: {confidence}")
    print(f"    what would make it wrong: {risk}\n")

benchmark (striking-distance weighted CTR): 0.347%

 1. content_5fe46e04994d | action: boost_ranking_priority | reason: quick_win_candidate
    why:        517,715 impressions/90d at avg position 4.2 (page_1) with 0.14% CTR
    confidence: low confidence - CTR gap may be intent/relevance, not just rank
    what would make it wrong: CTR is well below the 0.35% benchmark for its tier - if that's because the query/intent match is wrong (not just rank), pushing rank won't fix it

 2. content_aaef01a50def | action: boost_ranking_priority | reason: quick_win_candidate
    why:        517,109 impressions/90d at avg position 5.4 (page_1) with 0.25% CTR
    confidence: high confidence - typical page for its rank
    what would make it wrong: CTR is close to the 0.35% tier benchmark - a normal page for its rank, so the score here is really just volume, not an efficiency problem

 3. content_1a9e894be2e2 | action: boost_ranking_priority | reason: quick_win_candidate
    why:        416,180 impres

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

weak = top10[top10["ctr"] > benchmark_ctr * 1.3]
print("weak picks (already outperforming the tier benchmark, so less real headroom than their score suggests):")
print(weak[["rank", "content_id", "position_tier", "impressions_90d", "ctr"]].to_string(index=False))

print("\nwhy they're still ranked this high: the score is pure impression volume among qualifying rows - it")
print("doesn't discount for a page already converting well. that's a real weakness of a 1-signal score: it")
print("finds the BIGGEST pages in range, not necessarily the most INEFFICIENT ones. week 5's model should be")
print("able to do better than raw volume here.")

print("\n--- leakage check ---")
used_cols = {"position_tier", "impression_tier", "impressions_90d"}
forbidden = {"trend_direction", "trend_pct", "is_declining_label",
             "health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win"}
print("columns the score actually reads from:", sorted(used_cols))
print("none of these are label-derived or future-window:", used_cols.isdisjoint(forbidden))
print("none of FlyRank's own product flags exist in this dataset at all (checked against the data dictionary),")
print("and none were referenced by name in the scoring code above.")
assert used_cols.isdisjoint(forbidden), "leakage: a forbidden column made it into the score"
print("\nleakage check passed.")

weak picks (already outperforming the tier benchmark, so less real headroom than their score suggests):
 rank           content_id position_tier  impressions_90d  ctr
    4 content_2c2606c5d176        page_1           347399 0.53
    8 content_8e7ba84a972b        page_1           288426 0.92
    9 content_89e84d699e9e        page_1           275226 0.89
   10 content_aa4baf490b43        page_1           256290 0.50

why they're still ranked this high: the score is pure impression volume among qualifying rows - it
doesn't discount for a page already converting well. that's a real weakness of a 1-signal score: it
finds the BIGGEST pages in range, not necessarily the most INEFFICIENT ones. week 5's model should be
able to do better than raw volume here.

--- leakage check ---
columns the score actually reads from: ['impression_tier', 'impressions_90d', 'position_tier']
none of these are label-derived or future-window: True
none of FlyRank's own product flags exist in this dataset at all (

## Self-check

Before you submit, confirm each line honestly:

- [v] Every section above is filled — markdown thinking AND the code that backs it
- [v] The notebook runs top to bottom with no errors (Runtime → Run all)
- [v] No client names, URLs, or private queries anywhere
- [v] My claims use careful words: observed, measured, directional, decision-support
- [v] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.